# Setup & Environment Installation

In [ ]:
# Install required libraries
!pip install --upgrade google-cloud-storage google-cloud-bigquery pandas scikit-learn requests gcsfs sqlalchemy psycopg2-binary

In [ ]:
#!pip install -r requirements.txt

In [ ]:
#!pip install -q -r https://raw.githubusercontent.com/katiedevane/tennis-project/main/requirements.txt

#Imports

In [ ]:
import os
import requests
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from google.cloud import storage
from google.cloud import bigquery
from sqlalchemy import create_engine

# GCP Authentication & Configuration

---



In [ ]:
# --- GCP Configuration ---
from google.colab import userdata

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]

os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

# --- Authentication ---
from google.colab import auth
auth.authenticate_user()

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

In [ ]:
!gcloud services enable storage.googleapis.com bigquery.googleapis.com

In [ ]:
!gcloud services list --enabled | grep -E "storage|bigquery"

#Check intended bucket name is available

In [ ]:
from google.api_core.exceptions import Forbidden

client = storage.Client(project=PROJECT_ID)

try:
    existing = client.lookup_bucket(BUCKET_NAME)
    if existing:
        print("Bucket already created")
    else:
        print("Name appears available.")
except Forbidden:
    print("Bucket name already exists. Generate a new one, update TENNIS_BUCKET_NAME in Secrets, re-run Configuration & GCP Authentication, then re-run this cell.")

#Create bucket

In [ ]:
from google.api_core.exceptions import Conflict

bucket = storage.Bucket(client, BUCKET_NAME)
bucket.iam_configuration.uniform_bucket_level_access_enabled = True
bucket.iam_configuration.public_access_prevention = "enforced"

try:
    client.create_bucket(bucket, location=REGION)
    print("Created bucket in", REGION)
except Conflict as e:
    print("Not created:", e)

# Check bucket settings

In [ ]:
b = client.get_bucket(BUCKET_NAME)
print("location:", b.location)
print("public access prevention:", b.iam_configuration.public_access_prevention)
print("uniform access:", b.iam_configuration.uniform_bucket_level_access_enabled)

#Edit bucket protection settings

In [ ]:
b = client.get_bucket(BUCKET_NAME)

# Keep old versions of overwritten or deleted files...
b.versioning_enabled = True

# ...but delete old versions after 30 days, to keep storage cost bounded.
# (This replaces any existing rules, so re-running it doesn't add duplicates.)
bucket.clear_lifecycle_rules()
bucket.add_lifecycle_delete_rule(is_live=False, number_of_newer_versions=3)

b.patch()
print("Protections updated.")

In [ ]:
b = client.get_bucket(BUCKET_NAME)
print("versioning:", b.versioning_enabled)
print("lifecycle rules:", list(b.lifecycle_rules))
print("public access prevention:", b.iam_configuration.public_access_prevention)
print("uniform access:", b.iam_configuration.uniform_bucket_level_access_enabled)

sd = getattr(b, "soft_delete_policy", None)
if sd is not None:
    print("soft delete (days):", (sd.retention_duration_seconds or 0) / 86400)
else:
    print("soft delete: your library version can't show this; check it in the console")

#Connect project to BigQuery

In [ ]:
bq = bigquery.Client(project=PROJECT_ID, location=REGION)

for name in DATASETS:
    dataset = bigquery.Dataset(f"{PROJECT_ID}.{name}")
    dataset.location = REGION
    dataset.labels = {"project": "tennis-ml"}
    bq.create_dataset(dataset, exists_ok=True)

for d in bq.list_datasets():
    full = bq.get_dataset(d.reference)
    print(full.dataset_id, "|", full.location)

# Get tennis data into GCS bucket

 Download wta_matches_2010..2026 from mirror commit to the GCS bucket

In [ ]:
import hashlib
import json
import requests
from datetime import datetime, timezone
from google.cloud import storage

COMMIT = "83733587353df8a41f2fd4f516147d5aa83f5a8d"   # pins the exact snapshot
YEARS = range(2010, 2027)                           # 2010 through 2026

BASE_URL = f"https://raw.githubusercontent.com/Aneeshers/tennis-sackmann-archive/{COMMIT}/wta"
FOLDER = f"raw/tennis_sackmann_archive/{COMMIT[:8]}/wta"

bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)
manifest = []

for year in YEARS:
    name = f"wta_matches_{year}.csv"
    url = f"{BASE_URL}/{name}"

    # 1. Download
    response = requests.get(url, timeout=60)
    if response.status_code == 404:
        print(f"{name}: not found")
        continue
    response.raise_for_status()
    data = response.content

    # 2. Upload (skip if it's already there, so raw data is never overwritten)
    blob = bucket.blob(f"{FOLDER}/{name}")
    if blob.exists():
        status = "already stored"
    else:
        blob.upload_from_string(data, content_type="text/csv")
        status = "uploaded"

    # 3. Record what we got
    sha256 = hashlib.sha256(data).hexdigest()
    rows = data.count(b"\n") - 1
    manifest.append({"file": name, "url": url, "rows": rows, "sha256": sha256})
    print(f"{name}: {status} | {rows:,} rows | {sha256[:12]}")

# 4. Save the manifest (a timestamped record of this run)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
manifest_path = f"manifests/wta_matches_{COMMIT[:8]}_{stamp}.json"
bucket.blob(manifest_path).upload_from_string(
    json.dumps({"commit": COMMIT, "files": manifest}, indent=2),
    content_type="application/json",
)
print("Manifest saved:", manifest_path)

In [ ]:

blob = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME).blob(
    "manifests/wta_matches_83733587_20260929T173941Z.json"
)
print(blob.download_as_text())

In [ ]:

# Load configuration from Colab Secrets
GCS_RAW_FOLDER = userdata.get("GCS_RAW_FOLDER")

# Dataset configuration
RAW_URL = (
    "https://raw.githubusercontent.com/"
    "Aneeshers/tennis-sackmann-archive/"
    "main/wta/wta_matches_2009.csv"
)

LOCAL_RAW_PATH = "/content/wta_matches_2009.csv"


# Download
response = requests.get(RAW_URL, timeout=30)
response.raise_for_status()

with open(LOCAL_RAW_PATH, "wb") as f:
    f.write(response.content)

print(f"Downloaded: {LOCAL_RAW_PATH}")


# Upload
client = storage.Client()
bucket = client.bucket(BUCKET_NAME)

destination_blob_name = (
    f"{GCS_RAW_FOLDER}/wta_matches_2009.csv"
)

blob = bucket.blob(destination_blob_name)
blob.upload_from_filename(LOCAL_RAW_PATH)

print(
    f"Uploaded -> "
    f"gs://{BUCKET_NAME}/{destination_blob_name}"
)


Check that all .csv files have the same columns in same order, because BigQuery matches columns by position, so the header rows need to match

In [ ]:


gcs = storage.Client(project=PROJECT_ID)
RAW_PREFIX = "raw/tennis_sackmann_archive/83733587/wta"

headers = {}
for blob in gcs.list_blobs(BUCKET_NAME, prefix=f"{RAW_PREFIX}/wta_matches_20"):
    first_line = blob.download_as_bytes(start=0, end=4095).decode("utf-8-sig").splitlines()[0]
    headers[blob.name.split("/")[-1]] = first_line

distinct = set(headers.values())
print(len(headers), "files,", len(distinct), "distinct header row(s)")
assert len(distinct) == 1, "Headers differ between files - stop and tell me"

COLUMNS = next(iter(distinct)).split(",")
print(len(COLUMNS), "columns:", COLUMNS)

Load tennis data into BiqQuery

In [ ]:

TABLE = f"{PROJECT_ID}.tennis_raw.wta_matches"
uri = f"gs://{BUCKET_NAME}/{RAW_PREFIX}/wta_matches_20*.csv"

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    schema=[bigquery.SchemaField(c, "STRING") for c in COLUMNS],
    write_disposition="WRITE_TRUNCATE",
    max_bad_records=0,          # any malformed row makes the job fail loudly
)
job = bq.load_table_from_uri(uri, TABLE, job_config=job_config)
job.result()
print("Load job:", job.job_id)
print("Rows loaded:", job.output_rows)

Checking that BigQuery row counts match data download (ensure nothing has been lost in transit)

In [ ]:
sql = f"""
SELECT SUBSTR(tourney_date, 1, 4) AS year, COUNT(*) AS matches
FROM `{TABLE}`
GROUP BY year ORDER BY year
"""
for row in bq.query(sql).result():
    print(row.year, row.matches)